# 00 · EDA — Jena Climate & UCI HAR

두 시계열 데이터셋을 탐색하고 **무작위 샘플을 즉시 확인**할 수 있는 노트북입니다.

- **Jena Climate** — 독일 막스플랑크 기상관측소, 10분 간격 14변수
- **UCI HAR** — 스마트폰 가속도/자이로 6가지 활동 분류 (128 timestep × 9 channel)

> ⚠️ 무작위 샘플 셀은 `seed=None`이라 **재실행할 때마다 다른 샘플**이 나옵니다. 같은 결과를 보고 싶으면 셀에서 `seed=42`처럼 지정하세요.

## 0. 환경 점검

In [ ]:
import sys, platform
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('Python      :', platform.python_version())
print('Torch       :', torch.__version__)
print('CUDA avail  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device      :', torch.cuda.get_device_name(0))
    print('CUDA build  :', torch.version.cuda)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import data_loader, visualize

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

## 1. Jena Climate

최초 1회만 다운로드 (~13MB). 이후 `data/raw/` 캐시 사용.

In [ ]:
df = data_loader.load_jena_climate()
print('shape   :', df.shape)
print('range   :', df.index.min(), '→', df.index.max())
print('missing :', int(df.isna().sum().sum()))
df.head()

In [ ]:
df.dtypes

In [ ]:
df.describe().T

### 1.1 전체 기온 시계열 (다운샘플 시각화)

10분 간격이 너무 촘촘해서 6시간 평균으로 다운샘플링한 뒤 plot.

In [ ]:
ax = df['T (degC)'].resample('6h').mean().plot(figsize=(12, 3), lw=0.6)
ax.set_title('Jena Climate — Temperature (6-hour mean)')
ax.set_ylabel('T (degC)')
plt.show()

### 1.2 ★ 무작위 샘플 뷰어

아래 셀들을 **여러 번 재실행**해 보세요 — 매번 다른 시점 / 다른 행이 나옵니다.

In [ ]:
# 무작위로 10개 행 보기 (실제 데이터가 어떤 모양인지)
df.sample(10).sort_index()

In [ ]:
# 무작위 7일 구간의 4개 변수 시계열
_ = visualize.plot_random_climate_window(
    df, days=7,
    columns=('T (degC)', 'rh (%)', 'wv (m/s)', 'p (mbar)'),
    seed=None,
)
plt.show()

In [ ]:
# 무작위 1일 구간의 기온/습도
_ = visualize.plot_random_climate_window(
    df, days=1,
    columns=('T (degC)', 'rh (%)'),
    seed=None,
)
plt.show()

### 1.3 계절성 분해 (월 평균 기온)

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

monthly_T = df['T (degC)'].resample('1M').mean().dropna()
result = seasonal_decompose(monthly_T, model='additive', period=12)
fig = result.plot()
fig.set_size_inches(11, 7)
plt.show()

### 1.4 ACF / PACF (시간 단위 다운샘플)

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

hourly = df['T (degC)'].resample('1h').mean().dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
plot_acf(hourly, lags=72, ax=axes[0])  # 3 days
plot_pacf(hourly, lags=72, ax=axes[1], method='ywm')
plt.show()

## 2. UCI HAR (Human Activity Recognition)

최초 1회만 다운로드 (~60MB). 9채널(가속도 x/y/z, 자이로 x/y/z, 합산 가속 x/y/z) × 128 timestep 윈도우.

In [ ]:
har = data_loader.load_uci_har()
print('X_signals_train :', har['X_signals_train'].shape)
print('X_features_train:', har['X_features_train'].shape)
print('y_train         :', har['y_train'].shape, '— uniques:', sorted(set(har['y_train'].tolist())))
print('X_signals_test  :', har['X_signals_test'].shape)
print('y_test          :', har['y_test'].shape)
print('label map       :', har['label_names'])
print('channels        :', har['channel_names'])

### 2.1 클래스 분포

In [ ]:
_ = visualize.plot_har_class_distribution(
    har['y_train'], har['label_names'], title='HAR train class distribution'
)
plt.show()

### 2.2 ★ 무작위 샘플 뷰어

아래 셀들을 여러 번 재실행해 보세요.

In [ ]:
# 무작위 5개 윈도우 — 9채널 모두 한 subplot에 겹쳐서
_ = visualize.plot_random_har_samples(
    har['X_signals_train'], har['y_train'],
    label_names=har['label_names'], channel_names=har['channel_names'],
    n=5, seed=None,
)
plt.show()

In [ ]:
# 무작위 클래스 → 무작위 윈도우 → 9채널 stack plot
_ = visualize.plot_random_har_class_signals(
    har['X_signals_train'], har['y_train'],
    label_names=har['label_names'], channel_names=har['channel_names'],
    seed=None,
)
plt.show()

### 2.3 활동별 평균 |signal| heatmap

각 활동(class)이 어떤 채널에서 큰 값을 보이는지 한눈에 파악.

In [ ]:
_ = visualize.plot_har_mean_signal_heatmap(
    har['X_signals_train'], har['y_train'],
    label_names=har['label_names'], channel_names=har['channel_names'],
)
plt.show()

### 2.4 561개 handcrafted feature 일부 분포

In [ ]:
X_feat = har['X_features_train']
feature_names = har['feature_names']
rng = np.random.default_rng(None)
sel = rng.choice(X_feat.shape[1], size=6, replace=False)
fig, axes = plt.subplots(2, 3, figsize=(11, 5))
for ax, idx in zip(axes.flat, sel):
    ax.hist(X_feat[:, idx], bins=40, color='steelblue', alpha=0.85)
    ax.set_title(feature_names[idx][:30], fontsize=8)
plt.tight_layout()
plt.show()

## 3. 결론 메모

여기에 EDA에서 발견한 점을 자유롭게 적어주세요.

- 예) Jena 기온은 강한 연 단위 계절성, 일 단위 진동도 뚜렷 → SARIMA에 (period=24, period=8760) 시도 가치
- 예) HAR에서 LAYING 활동은 가속도가 거의 0에 가까움 → CNN이 쉽게 분리할 듯
- ...